# 实践项目 05：空间转录组表达超分辨率

本 Notebook 使用课程准备的配对 H&E、LR、HR 和 split 数据。LR 表示 16 μm Snap25 输入，HR 表示 2 μm 参考表达图，模型学习在细网格上估计一个高表达基因的局部表达。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码中用整行注释标出了需要填写的位置。先阅读当前单元格的输入、处理和输出，再修改标记区域。

## 任务总览

1. 核对四个字段的 shape、split 数量和 Snap25 表达范围。
2. 选择中心区域，查看同一位置的大图、小图、H&E、LR 和 HR。
3. 补全 H&E 与粗尺度表达联合输入的轻量网络。
4. 完成 log 空间损失和 8×8 区域总量约束。
5. 比较模型与插值基线的 MAE、相关性、聚合误差和空间图。

## 需要保存的结果

`task5_data_visualization.png`、`task5_scale_overview.png`、`task5_training_curve.png`、`task5_prediction_visualization.png`、`task5_result.json`。


In [ ]:
from pathlib import Path  # 导入当前步骤需要的工具
import json, random  # 导入当前步骤需要的工具
import numpy as np  # 导入当前步骤需要的工具
import matplotlib.pyplot as plt  # 导入当前步骤需要的工具
import torch  # 导入当前步骤需要的工具
import torch.nn as nn  # 导入当前步骤需要的工具
from torch.utils.data import Dataset, DataLoader  # 导入当前步骤需要的工具

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 固定随机状态以便复现实验
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 保存当前步骤使用的中间结果
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)  # 保存输出文件目录
DATA_PATH = None  # 可在本地填写课程 NPZ 路径；Kaggle 自动查找固定文件名
candidates = sorted(Path('/kaggle/input').rglob('kydw-try-a05-paired-patches.npz'))  # 读取本任务需要的数据
if DATA_PATH is None and candidates: DATA_PATH = candidates[0]  # 根据当前条件选择处理分支
assert DATA_PATH is not None, '请挂载包含 kydw-try-a05-paired-patches.npz 的课程数据集。'  # 执行当前步骤并保留结果
data = np.load(DATA_PATH, allow_pickle=True)  # 读取本任务需要的数据
required = {'he','lr','hr','split'}  # 核对输入字段
assert required.issubset(data.files), required - set(data.files)  # 执行当前步骤并保留结果
he = data['he'].astype(np.float32) / 255.0  # 读取 H&E 图像并归一化
lr = data['lr'].astype(np.float32)  # 读取 16 μm 粗尺度表达总量
hr = data['hr'].astype(np.float32)  # 读取 2 μm 参考表达图
split = data['split'].astype(str)  # 读取预先划分的数据集合
print('he/lr/hr:', he.shape, lr.shape, hr.shape, 'splits:', {k: int((split == k).sum()) for k in np.unique(split)})  # 显示核对结果


## 任务 1：核对输入字段与空间划分

H&E、LR 和 HR 覆盖同一空间区域。LR 在每个 8×8 区域内记录粗尺度总量；`split` 是课程数据预先提供的空间划分。


In [ ]:
# ===== 项目05·任务1·学生填写区（开始） =====
# TODO：统计各 split 数量、shape、非零比例和最大值，并确认 split 只包含 train、validation、test。
summary = None  # 保存当前步骤的核对结果
# ===== 项目05·任务1·学生填写区（结束） =====
print(summary)  # 显示便于检查的关键信息


## 任务 2：配对大图、小图与表达图

从 train 样本中选择组织覆盖较完整、表达信号可见的一项，把 H&E、LR 粗尺度密度和 HR 参考表达图放在同一行，再截取中心区域放大。三列来自同一个空间区域，色标可以分别设置。


In [ ]:
# ===== 项目05·任务2·学生填写区（开始） =====
# TODO：选择中心组织覆盖完整且 HR 非零比例较高的 train 样本，绘制大图与中心小图。
sample_index = None  # 填写一个 train 样本的整数索引
# ===== 项目05·任务2·学生填写区（结束） =====
print('请完成同一区域的大图、小图、H&E、LR 和 HR 可视化。')  # 提示当前任务的输出


## 任务 3：补全融合网络

输入为 4 个通道（H&E 三通道和 LR 密度），输出为 1 个通道的细尺度表达预测。


In [ ]:
# ===== 项目05·任务3·学生填写区（开始） =====
class STDataset(Dataset):  # 定义本任务使用的数据结构
    def __init__(self, kind): self.ids = np.where(split == kind)[0]  # 保存指定集合的样本编号
    def __len__(self): return len(self.ids)  # 返回样本数量
    def __getitem__(self, k):  # 读取一个样本
        i = int(self.ids[k]); lr_total = lr[i]  # 读取样本和粗尺度总量
        x = np.concatenate([he[i], np.log1p(lr_total / 64.0)], axis=0)  # 拼接四通道输入
        y = np.log1p(hr[i])  # 使用 log1p 压缩稀疏表达范围
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(lr_total), i  # 返回输入、目标和编号

class SRNet(nn.Module):  # 定义本任务使用的模型
    def __init__(self):  # 初始化网络
        super().__init__()  # 初始化父类
        # TODO：补全 4 通道输入、1 通道输出的卷积结构。
        self.body = None  # 保存模型主体
    def forward(self, x): return torch.nn.functional.softplus(x[:, 3:4] + self.body(x))  # 保持非负输出

model = None  # TODO：实例化 SRNet 并移动到 DEVICE
# ===== 项目05·任务3·学生填写区（结束） =====
print(model)  # 显示便于检查的模型结构


## 任务 4：损失和 8×8 总量约束

预测值按 8×8 区域求和后，应与 LR 中的粗尺度总量接近；这项约束让输出保留观测到的总量。


In [ ]:
# ===== 项目05·任务4·学生填写区（开始） =====
def aggregate8(x): return torch.nn.functional.avg_pool2d(x, 8, 8) * 64  # 计算每个 8×8 区域的总量

def loss_fn(pred_log, target_log, lr_raw):  # 定义训练损失
    pred = torch.expm1(pred_log).clamp_min(0)  # 把预测恢复到原始计数尺度
    target = torch.expm1(target_log).clamp_min(0)  # 把目标恢复到原始计数尺度
    l1 = (pred_log - target_log).abs().mean()  # 计算 log 空间误差
    # TODO：计算 aggregate8(pred) 与 avg_pool2d(lr_raw, 8, 8) 的差异。
    consistency = None  # 保存聚合一致性损失
    return l1 + .1 * consistency  # 合并逐像素和区域级损失
# ===== 项目05·任务4·学生填写区（结束） =====
print('请完成 8×8 聚合一致性损失。')  # 提示当前任务的输出


## 任务 5：训练与结果比较

完成训练后，比较模型与插值基线的 MAE、Pearson 相关性和 8×8 聚合误差，并查看预测、参考和误差图。


In [ ]:
# ===== 项目05·任务5·学生填写区（开始） =====
# TODO：完成训练、验证选模、模型与插值基线比较，以及结果保存。
# 参考答案会使用较长训练和稳定的模型设置，但题目版不直接给出完整结果。
# ===== 项目05·任务5·学生填写区（结束） =====
print('输出文件应写入', OUT)  # 显示便于检查的关键信息
